# Maya: healthcare operations LLM — training & evaluation

Non-clinical hospital operations only. No diagnosis, treatment, prescription or triage.

Runtime: **Colab T4** (16GB). Steps below: clone repo, install pinned deps, validate the exact checkpoint, prepare the licensed dataset, run the baseline, smoke-test training, train, evaluate on the locked test set, produce artifacts, save to Drive.

Model: `empero-ai/Qwen3.8-4B-Distill` (Qwen3.5 architecture, Apache-2.0). Framework: Unsloth.

## 1. Clone the repository

In [ ]:
import os
REPO = os.environ.get("MAYA_REPO", "https://github.com/Cyrax321/maya-healthcare-llm.git")
BRANCH = "main"
!rm -rf maya-healthcare-llm
!git clone -b {BRANCH} {REPO}
%cd maya-healthcare-llm
import sys
sys.path.insert(0, os.path.abspath('src'))  # make the maya package importable right away

## 2. Install pinned dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps unsloth_zoo
# Qwen3.5 requires transformers v5, but unsloth_zoo caps at <=5.17.0 (excludes 5.0.0/5.1.0).
# Pin 5.15.0: matches the model's generation_config and unsloth_zoo's allowed range.
!pip install -q "transformers==5.15.0"
# NOTE: flash-linear-attention/causal-conv1d are only needed for non-Unsloth
# inference (vLLM/plain transformers). Unsloth ships its own Mamba/DeltaNet Triton
# kernels for training + inference, so we skip the slow source builds here.
!pip install -e . --no-deps
# fallback: make the package importable even if the editable install is flaky
import sys, os
sys.path.insert(0, os.path.abspath('src'))
import maya
print('maya importable:', maya.__version__)

## 3. GPU / RAM / software diagnostics

In [ ]:
import sys, torch, transformers
print("python", sys.version.split()[0])
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("transformers", transformers.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("vram GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## 4. Validate the exact checkpoint + tokenizer

In [ ]:
from maya.model import inspect_checkpoint, DEFAULT_REPO
from maya.config import resolve_model_revision
print(inspect_checkpoint(DEFAULT_REPO))
print("resolved revision:", resolve_model_revision(DEFAULT_REPO))

## 5. Audit + prepare the licensed dataset

In [ ]:
from maya import data
manifest = data.prepare_dataset(seed=20241015, examples_per_category=520, out_dir="data/processed")
import json
print(json.dumps(manifest, indent=2))

## 6. Original-model baseline (E0)

In [ ]:
# E0 runs the FULL locked test set so E0 and E1 use identical cases for the paired comparison.
!PYTHONPATH=src python -m maya.cli evaluate --experiment-name E0 --experiment-id E0

## 7. Training smoke test (proves load + fit before full run)

In [ ]:
import json
from maya import config, data, train
from maya.model import DEFAULT_REPO
cfgs = config.load_all_configs()
cfgs["training_t4"]["training"]["max_steps"] = 5  # smoke: just prove it loads and fits
tr = data.load_processed_split("data/processed", "train")[:64]
va = data.load_processed_split("data/processed", "validation")[:16]
res = train.run_training(DEFAULT_REPO, tr, va, "outputs/smoke", cfgs, resume=False)
print("smoke train OK:", res.final_checkpoint)

# free GPU memory so the full-train subprocess in the next cell can load the model again
import gc, torch
del res
gc.collect()
torch.cuda.empty_cache()
print("GPU freed after smoke train")

## 8. Configured training run (resumable)

In [ ]:
!PYTHONPATH=src python -m maya.cli train --output-dir outputs/maya-q1

## 9. Evaluate base vs adapter on the same locked test set (E0/E1)

In [ ]:
# E1: fine-tuned adapter. Point --adapter at the saved LoRA dir.
!PYTHONPATH=src python -m maya.cli evaluate --experiment-name E1 --experiment-id E1 --adapter outputs/maya-q1

## 9b. Statistical comparison E0 vs E1 (paired McNemar + bootstrap CI)

In [ ]:
!PYTHONPATH=src python -m maya.cli compare --baseline artifacts/E0 --adapter artifacts/E1

## 10. Reports, plots and metrics (per experiment id)

In [ ]:
# Reports/plots/metrics are written by the evaluate step to artifacts/<id>/.
!ls -R artifacts | head -50

## 11. Save adapters + reports to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil, os, glob
DEST = '/content/drive/MyDrive/maya'
# full experiment results: every metrics.json, metrics.csv, report.md,
# predictions.jsonl, errors.jsonl and every figure (PNG + SVG) per experiment
shutil.copytree('artifacts', f'{DEST}/artifacts', dirs_exist_ok=True)
# trained adapters / checkpoints
shutil.copytree('outputs', f'{DEST}/outputs', dirs_exist_ok=True)
print('\n=== saved to Drive:', DEST, '===')
for exp in sorted(os.listdir('artifacts')):
    if exp == 'README.md':
        continue
    png = len(glob.glob(f'artifacts/{exp}/plots/*.png'))
    svg = len(glob.glob(f'artifacts/{exp}/plots/*.svg'))
    print(f'{exp}: {png} PNG + {svg} SVG plots, metrics.json/csv, report.md, predictions.jsonl, errors.jsonl')

## 12. Push model + results to Hugging Face Hub

Token comes from a Colab Secret named `HF_TOKEN` (never paste it into a cell).
Add it under the key icon (Secrets) in the left panel, then grant notebook access.

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # secret, never committed
HUB_USER = os.environ.get('HUB_USER', 'Cyrax321')   # change to your HF username

In [ ]:
# Push results (metrics/predictions/plots/report) as a private dataset repo
!PYTHONPATH=src python -m maya.cli push --what results --hub-repo-id {HUB_USER}/Maya-Q1-results \
    --experiment-id $(ls -1 artifacts | grep -v README | tail -1) --private

# Push the trained adapter as the Maya-Q1 model repo
!PYTHONPATH=src python -m maya.cli push --what model --hub-repo-id {HUB_USER}/Maya-Q1 \
    --adapter-dir outputs/maya-q1 --private